# Practica: NLP - Entrenamiento/Test

Alumno: Hannibal Tomasson Izquierdo

Instrucciones: El alumno, con los datos preprocesados del ejercicio 2, deberá entrenar dos
modelos distintos de los que, tras comparar sus resultados, elegirá uno como el mejor.
Para tomar esta decisión se basará en las métricas que calcule (precision, recall, f1-score,
...). El enfoque será el de un problema de clasificación binaria supervisada.
Los modelos deberán tomar a su entrada los datos codificados con un modelo de
bolsa de palabras (bag-of-words). Se deberán justificar los parámetros del vectorizer, así
como tener en cuenta aspectos como el balanceo de clases.
La elección de los modelos es libre.

En esta etapa vamos a proceder a entrenar varios modelos con el dataset que hemos preprocesado en el paso anterior. Inicialmente vamos a entrenar dos modelos de Machine Learning, uno de Regresion Logistica, y uno de Random Forest. Posteriormente, vamos a intentar entrenar tambien un modelo utilizando tecnicas de Deep Learning.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/Colab Notebooks/NLP/Practica/reviews_amazon_books_balanced_preprocessed.csv"

df = pd.read_csv(file_path)

In [4]:
# Comprobamos que se ha cargado correctamente

print(df.shape)
df[['reviewText', 'clean_text', 'sentiment']].head()

(5000, 11)


,reviewText,clean_text,sentiment
0,At the beginning of each chapter it switches b...,beginning chapter switch anna james find neat ...,neg
1,"Historically inaccurate, a painfully rushed th...",historically inaccurate painfully rush third a...,neg
2,I've spent 5 months reading these books. Each ...,ive spend month read book book first become sl...,neg
3,Can honestly say this is the first book of Goo...,honestly say first book goodkinds move archive...,neg
4,Note -- Tolkien's work is rated 5 stars. The p...,note tolkien work rat star physical book rat t...,neg


## Machine Learning

Para transformar el texto a representaciones numericas vamos a utilizar la tecnica de Bag of Words, mediante CountVectorizer. Vamos a limitar el vectorizador a las 5000 palabras mas frecuentes, para asi reducir la dimensionalidad y su coste computacional. Ademas, haciendo esto evitamos el sobreajuste que podria surgir si introducimos terminos muy infrecuentes o ruido.

El resto de parametros se mantuvieron a sus valores por defecto, ya que en el paso anterior ya habiamos prepocesado los textos.


In [27]:
from sklearn.feature_extraction.text import CountVectorizer

# Inicializamos el vectorizador
vectorizer = CountVectorizer(max_features=5000)

# Ajustamos el vectorizador a los datos de entrenamiento
X = vectorizer.fit_transform(df['clean_text'])

# Variables predictoras
X.shape

(5000, 5000)

In [28]:
# Codificamos la variable objetivo 'sentiment'
y = df['sentiment'].map({'pos': 1, 'neg': 0})

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Tamaño del set de entrenamiento:", X_train.shape)
print("Tamaño del set de test:", X_test.shape)

Tamaño del set de entrenamiento: (4000, 5000)
Tamaño del set de test: (1000, 5000)


In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Entreamos el modelo de regresion logistica
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)

#Entrenamos el modelo random forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

In [32]:
from sklearn.metrics import classification_report

# Predicciones
y_pred_logreg = logreg.predict(X_test)
y_pred_rf = rf.predict(X_test)

# Generamos los informes

print("Informe Regresion Logistica")
print(classification_report(y_test, y_pred_logreg, target_names=['neg', 'pos']))

print("\nInforme Random Forest")
print(classification_report(y_test, y_pred_rf, target_names=['neg', 'pos']))

Informe Regresion Logistica
              precision    recall  f1-score   support

         neg       0.83      0.83      0.83       500
         pos       0.83      0.83      0.83       500

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83      1000
weighted avg       0.83      0.83      0.83      1000


Informe Random Forest
              precision    recall  f1-score   support

         neg       0.74      0.84      0.79       500
         pos       0.82      0.71      0.76       500

    accuracy                           0.78      1000
   macro avg       0.78      0.78      0.78      1000
weighted avg       0.78      0.78      0.78      1000



Segun lo que podemos observar en los informes de clasificacion, el modelo de Regresion Logistica es el que mejor rendimiento tiene. Es equilibrado y tiene buena capacidad de generalizacion.

In [34]:
import numpy as np

# Guardamos las etiquetas y predicciones de los modelos de ML
np.save("/content/drive/MyDrive/Colab Notebooks/NLP/Practica/y_true_ml.npy", y_test)
np.save("/content/drive/MyDrive/Colab Notebooks/NLP/Practica/y_pred_logreg.npy", y_pred_logreg)
np.save("/content/drive/MyDrive/Colab Notebooks/NLP/Practica/y_pred_rf.npy", y_pred_rf)

print("Etiquetas y Predicciones Guardadas")

Etiquetas y Predicciones Guardadas


## Deep Learning

Ahora, para poder comparar el rendimiento de nuestros modelos entrenados con Machine Learning, vamos a crear una red neuronal recurrente basica y entrenarla con nuestro dataset.

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenizamos
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

# Transformamos el texto a secuencias
sequences = tokenizer.texts_to_sequences(df['clean_text'])

# Igualamos longitud con padding
X = pad_sequences(sequences, maxlen=200, padding='post', truncating='post')

In [11]:
y = df['sentiment'].map({'pos': 1, 'neg': 0}).values

In [12]:
from sklearn.model_selection import train_test_split

# Dividimos el dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.optimizers import Adam

# Definimos el modelo
model = Sequential([
    Embedding(input_dim=10000, output_dim=64, input_length=200),
    LSTM(64),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

model.summary()

Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 103ms/step - accuracy: 0.4874 - loss: 0.6948 - val_accuracy: 0.5140 - val_loss: 0.6930
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 19s 95ms/step - accuracy: 0.5084 - loss: 0.6927 - val_accuracy: 0.5170 - val_loss: 0.6925
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 21s 100ms/step - accuracy: 0.5367 - loss: 0.6810 - val_accuracy: 0.5230 - val_loss: 0.6951
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 13s 104ms/step - accuracy: 0.5576 - loss: 0.6501 - val_accuracy: 0.4920 - val_loss: 0.7162
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 19s 94ms/step - accuracy: 0.5475 - loss: 0.6406 - val_accuracy: 0.5150 - val_loss: 0.7227


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (32, 200, 64)          │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (32, 64)               │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (32, 1)                │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,019,269 (7.70 MB)

 Trainable params: 673,089 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,346,180 (5.14 MB)

In [15]:
model.evaluate(X_test, y_test)

32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.4904 - loss: 0.7334


[0.7226576805114746, 0.5149999856948853]

Como vemos, el resultado ha quedado muy por debajo del obtenido por nuestras tecnicas de ML, pero esto no es demasiado sorprendente por que la estructura que hemos usado era bastante poco avanzada.

Ahora que ya tenemos una linea base, vamos a mejorar nuestro modelo. Para ello vamos a emplear una arquitectura mas compleja, introduciendo capas con vectores preentrenados, asi como una LSTM (Long Short Term Memory) bidireccional, para asi capturar informacion contextual de lado a lado. Los vectores preentrenados proceden de GloVe, que contienen informacion semantica aprendida a partir de grandes corpus de texto. Adicionalmente, vamos a incluir una capa de Dropout para prevenir el sobreajuste.

In [16]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip

--2025-05-01 16:30:42--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-05-01 16:30:42--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-05-01 16:30:42--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [20]:
import numpy as np

# Cargamos el archivo GloVe

embeddings_index = {}
with open("glove.6B.100d.txt", encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

# Creamos la matrix de embeddings

embedding_dim = 100
num_words = 10000
embedding_matrix = np.zeros((num_words, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i < num_words:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector


In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.optimizers import Adam


# Construimos el modelo
model = Sequential([
    Embedding(input_dim=num_words,
              output_dim=embedding_dim,
              weights=[embedding_matrix],
              input_length=200,
              trainable=False),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(loss=binary_crossentropy, optimizer='adam', metrics=['accuracy'])


# Ejecutamos el modelo
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model.fit(X_train, y_train, validation_data=(X_test, y_test),
          epochs=10,
          batch_size=32,
          callbacks=[early_stop])

model.summary()

Epoch 1/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 27s 179ms/step - accuracy: 0.5891 - loss: 0.6681 - val_accuracy: 0.7290 - val_loss: 0.5633
Epoch 2/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 40s 172ms/step - accuracy: 0.7455 - loss: 0.5271 - val_accuracy: 0.7750 - val_loss: 0.4875
Epoch 3/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 39s 157ms/step - accuracy: 0.7819 - loss: 0.4790 - val_accuracy: 0.7550 - val_loss: 0.5098
Epoch 4/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 21s 169ms/step - accuracy: 0.8091 - loss: 0.4389 - val_accuracy: 0.7980 - val_loss: 0.4454
Epoch 5/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 40s 166ms/step - accuracy: 0.8118 - loss: 0.4172 - val_accuracy: 0.8050 - val_loss: 0.4359
Epoch 6/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 41s 165ms/step - accuracy: 0.8274 - loss: 0.3902 - val_accuracy: 0.8130 - val_loss: 0.4240
Epoch 7/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 40s 156ms/step - accuracy: 0.8366 - loss: 0.3732 - val_accuracy: 0.8010 - val_loss: 0.4255
Epoch 8/10
125/125 ━━━━━━━━━━━━━━━━━━━━ 21s 157ms/step - accuracy: 0.8394 - loss: 0

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (32, 200, 100)         │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (32, 128)              │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (32, 128)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,253,829 (4.78 MB)

 Trainable params: 84,609 (330.50 KB)

 Non-trainable params: 1,000,000 (3.81 MB)

 Optimizer params: 169,220 (661.02 KB)

In [25]:
import pandas as pd

# Guardamos el historial del modelo DL
df_history = pd.DataFrame(model.history.history)
df_history.to_csv("/content/drive/MyDrive/Colab Notebooks/NLP/Practica/dl_training_history.csv", index=False)

print("Historial de entrenamiento DL guardado.")

Historial de entrenamiento DL guardado.


In [26]:
import numpy as np

# Guardamos las predicciones
y_pred_dl = (model.predict(X_test) > 0.5).astype(int).flatten()
np.save("/content/drive/MyDrive/Colab Notebooks/NLP/Practica/y_pred_dl.npy", y_pred_dl)

print("Predicciones guardadas.")

32/32 ━━━━━━━━━━━━━━━━━━━━ 4s 113ms/step
Predicciones guardadas.


El modelo de DL ha alcanzado una precision de validacion del 81.3%, comparable con la regresion logistica (83%). Además, se observó una buena evolucion del aprendizaje con señales de overfitting moderado a partir del epoch 7, que se ha solucionado gracias al EarlyStopping.

Las predicciones y metricas de los modelos entrenados en esta etapa han sido guardados, y seran comparados en la Etapa 4.